# Mood Cartographer
Contextual music intelligence agent. Maps temporal + weather + calendar signals to mood state, returns playlist concept with reasoning. Prototype of the SiriusXM concept from the Sierra AI pitch.

In [ ]:
import os

os.environ["ANTHROPIC_API_KEY"] = "your-key-here"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "your-key-here"
os.environ["LANGSMITH_PROJECT"] = "mood-cartographer"

print("✅ Env set, tracing to project: mood-cartographer")

✅ Env set, tracing to project: mood-cartographer


In [3]:
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/calendar.readonly",
]

# Delete old token to force re-auth with new scope
if os.path.exists("token.json"):
    os.remove("token.json")

flow = InstalledAppFlow.from_client_secrets_file(
    os.path.expanduser("~/credentials.json"), SCOPES
)
google_creds = flow.run_local_server(port=0)

with open("token.json", "w") as f:
    f.write(google_creds.to_json())

calendar = build("calendar", "v3", credentials=google_creds)
gmail = build("gmail", "v1", credentials=google_creds)

# Sanity check
events = calendar.events().list(
    calendarId="primary",
    maxResults=3,
    singleEvents=True,
    orderBy="startTime",
    timeMin="2026-05-10T00:00:00Z"
).execute()

print(f"✅ Calendar: {len(events.get('items', []))} upcoming events")
for e in events.get("items", [])[:3]:
    print(f"  - {e.get('summary', '(no title)')} @ {e.get('start', {}).get('dateTime', e.get('start', {}).get('date'))}")

/Users/siddharthrallabandi/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/siddharthrallabandi/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/siddharthrallabandi/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Pleas

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=953505929222-oo46t0obv4bv25fl19j2s6sce6mv9g42.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A49812%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.readonly&state=rttjpSPU5UChB74CWMXVZlPfL8cg3r&code_challenge=_LwPgokznlfr__kkqLkaKGPxa0eJ3C7hwfGN7OS8qCk&code_challenge_method=S256&access_type=offline
✅ Calendar: 1 upcoming events
  - [CONFIRMED] AWS x GDS Live Dinner @ Harris' Restaurant , SF | Siddharth Rallabandi @ 2026-05-11T18:30:00-07:00


In [4]:
import requests
from langchain.tools import tool
from datetime import datetime

@tool
def get_weather(location: str = "San Francisco") -> str:
    """Get current weather and today's forecast for a location. Returns temperature, condition, and a mood-relevant descriptor (sunny/cloudy/rainy/etc.)."""
    # Geocode
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": location, "count": 1}
        ).json()
        if not geo.get("results"):
            return f"Location '{location}' not found"
        lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]
        city = geo["results"][0]["name"]

        # Weather
        w = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "current": "temperature_2m,weather_code,cloud_cover,wind_speed_10m",
                "temperature_unit": "fahrenheit",
                "timezone": "auto",
            }
        ).json()

        c = w["current"]
        codes = {
            0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
            45: "foggy", 48: "foggy",
            51: "light drizzle", 53: "drizzle", 55: "heavy drizzle",
            61: "light rain", 63: "rain", 65: "heavy rain",
            71: "light snow", 73: "snow", 75: "heavy snow",
            80: "rain showers", 81: "rain showers", 82: "heavy rain showers",
            95: "thunderstorm",
        }
        condition = codes.get(c["weather_code"], "unknown")

        # Mood descriptor
        if c["weather_code"] in [0, 1]:
            mood = "bright"
        elif c["weather_code"] in [2, 3, 45, 48]:
            mood = "muted"
        elif c["weather_code"] >= 51:
            mood = "gloomy"
        else:
            mood = "neutral"

        return f"{city}: {c['temperature_2m']}°F, {condition}, cloud cover {c['cloud_cover']}%, wind {c['wind_speed_10m']} mph. Mood descriptor: {mood}."
    except Exception as e:
        return f"Error: {e}"

# Test
print(get_weather.invoke({"location": "San Francisco"}))

San Francisco: 53.2°F, overcast, cloud cover 100%, wind 10.0 mph. Mood descriptor: muted.


In [5]:
!pip install pytz

Defaulting to user installation because normal site-packages is not writeable


In [6]:
from datetime import datetime, timedelta, timezone
import pytz

@tool
def get_temporal_context(timezone_name: str = "America/Los_Angeles") -> str:
    """Returns current time, day-of-week, time-of-day bucket, and inferred life-rhythm context."""
    tz = pytz.timezone(timezone_name)
    now = datetime.now(tz)
    hour = now.hour
    day = now.strftime("%A")
    is_weekend = day in ["Saturday", "Sunday"]

    # Time-of-day bucket
    if 5 <= hour < 9:
        bucket = "early morning"
    elif 9 <= hour < 12:
        bucket = "late morning"
    elif 12 <= hour < 14:
        bucket = "midday"
    elif 14 <= hour < 17:
        bucket = "afternoon"
    elif 17 <= hour < 20:
        bucket = "early evening"
    elif 20 <= hour < 23:
        bucket = "late evening"
    else:
        bucket = "late night"

    # Life rhythm inference
    rhythms = []
    if day == "Monday" and hour < 10:
        rhythms.append("Monday-morning startup")
    if day == "Friday" and hour >= 17:
        rhythms.append("Friday wind-down")
    if is_weekend and hour < 11:
        rhythms.append("weekend slow start")
    if day == "Sunday" and hour >= 18:
        rhythms.append("Sunday work-prep")
    if not is_weekend and 17 <= hour < 19:
        rhythms.append("weekday commute/transition")

    rhythm_str = "; ".join(rhythms) if rhythms else "no strong rhythm pattern"

    return f"{day} {now.strftime('%I:%M %p')} ({bucket}). Weekend: {is_weekend}. Life rhythm: {rhythm_str}."


@tool
def get_calendar_load(hours_ahead: int = 24) -> str:
    """Returns calendar density for the next N hours. Categorizes as 'light', 'moderate', 'heavy'. Lists event titles."""
    try:
        now = datetime.now(timezone.utc)
        end = now + timedelta(hours=hours_ahead)
        events = calendar.events().list(
            calendarId="primary",
            timeMin=now.isoformat(),
            timeMax=end.isoformat(),
            singleEvents=True,
            orderBy="startTime"
        ).execute().get("items", [])

        count = len(events)
        if count == 0:
            load = "light (no events)"
        elif count <= 2:
            load = "light"
        elif count <= 5:
            load = "moderate"
        else:
            load = "heavy"

        titles = [e.get("summary", "(untitled)") for e in events[:5]]
        return f"Next {hours_ahead}h: {count} events ({load}). Titles: {titles}"
    except Exception as e:
        return f"Error: {e}"


# Test both
print("TEMPORAL:", get_temporal_context.invoke({}))
print()
print("CALENDAR:", get_calendar_load.invoke({"hours_ahead": 24}))

TEMPORAL: Sunday 10:32 PM (late evening). Weekend: True. Life rhythm: Sunday work-prep.

CALENDAR: Next 24h: 1 events (light). Titles: ["[CONFIRMED] AWS x GDS Live Dinner @ Harris' Restaurant , SF | Siddharth Rallabandi"]


In [7]:
!pip install -q spotipy

In [8]:
import os
if os.path.exists(".spotify_cache"):
    os.remove(".spotify_cache")

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth

SPOTIFY_CLIENT_ID = "your-key-here"
SPOTIFY_CLIENT_SECRET = "your-key-here"

sp = spotipy.Spotify(
    auth_manager=SpotifyOAuth(
        client_id=SPOTIFY_CLIENT_ID,
        client_secret=SPOTIFY_CLIENT_SECRET,
        redirect_uri="http://127.0.0.1:8888/callback",
        scope="user-top-read user-read-recently-played playlist-read-private",
        cache_path=".spotify_cache",
        open_browser=True,
    )
)

# Sanity check
me = sp.current_user()
print(f"✅ Spotify: connected as {me['display_name']}")

top = sp.current_user_top_tracks(limit=5, time_range="short_term")
print(f"\nYour top 5 tracks (last 4 weeks):")
for i, t in enumerate(top["items"], 1):
    print(f"  {i}. {t['name']} — {t['artists'][0]['name']}")

HTTP Error for GET to https://api.spotify.com/v1/me/ with Params: {} returned 403 due to Active premium subscription required for the owner of the app. When the subscription status changes, it can take a few hours before requests are allowed again.


SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/me/:
 Active premium subscription required for the owner of the app. When the subscription status changes, it can take a few hours before requests are allowed again., reason: None

# Taste Profile

In [11]:
TASTE_PROFILE = {
    "user": "Siddharth",
    "anchor_artists": ["Tame Impala", "Khruangbin", "Linkin Park", "Metallica", "Cannons"],
    "mood_preferences": {
        "Focus": {
            "vibe": "instrumental, low-stim, no lyrics competing for attention",
            "examples": ["Tame Impala instrumentals", "Khruangbin (instrumental tracks)", "lo-fi study mixes"],
            "tempo": "60-90 BPM",
        },
        "Drive": {
            "vibe": "high-energy, momentum, motivation — gets you moving",
            "examples": ["Linkin Park 'Hybrid Theory' era", "Tame Impala 'Elephant'", "upbeat psych rock"],
            "tempo": "110-140 BPM",
        },
        "Workout": {
            "vibe": "aggressive, heavy, high-output — pure intensity",
            "examples": ["Metallica 'Master of Puppets'", "Linkin Park 'Meteora'", "thrash metal"],
            "tempo": "140-180 BPM",
        },
        "Wind-down": {
            "vibe": "mellow, decompression, mid-tempo grooves",
            "examples": ["Khruangbin", "slow Tame Impala", "warm psych-soul"],
            "tempo": "70-100 BPM",
        },
        "Melancholic": {
            "vibe": "moody, atmospheric, reflective — gloomy day soundtrack",
            "examples": ["Cannons", "slow Tame Impala ('Yes I'm Changing')", "dream pop"],
            "tempo": "60-95 BPM",
        },
        "Social": {
            "vibe": "warm, conversation-safe, broadly accessible groove",
            "examples": ["Khruangbin", "soft funk", "psych-soul instrumentals"],
            "tempo": "85-110 BPM",
        },
    },
    "context_overrides": {
        "rainy_or_foggy_weather": "lean toward Melancholic (Cannons territory)",
        "Sunday_evening": "lean toward Focus or Wind-down",
        "heavy_calendar_tomorrow": "lean toward Focus tonight (prep mode)",
        "Friday_evening": "lean toward Social or Drive",
        "gym_or_workout_planned": "Workout mood overrides everything",
    },
}

import json
print(json.dumps(TASTE_PROFILE, indent=2))

{
  "user": "Siddharth",
  "anchor_artists": [
    "Tame Impala",
    "Khruangbin",
    "Linkin Park",
    "Metallica",
    "Cannons"
  ],
  "mood_preferences": {
    "Focus": {
      "vibe": "instrumental, low-stim, no lyrics competing for attention",
      "examples": [
        "Tame Impala instrumentals",
        "Khruangbin (instrumental tracks)",
        "lo-fi study mixes"
      ],
      "tempo": "60-90 BPM"
    },
    "Drive": {
      "vibe": "high-energy, momentum, motivation \u2014 gets you moving",
      "examples": [
        "Linkin Park 'Hybrid Theory' era",
        "Tame Impala 'Elephant'",
        "upbeat psych rock"
      ],
      "tempo": "110-140 BPM"
    },
    "Workout": {
      "vibe": "aggressive, heavy, high-output \u2014 pure intensity",
      "examples": [
        "Metallica 'Master of Puppets'",
        "Linkin Park 'Meteora'",
        "thrash metal"
      ],
      "tempo": "140-180 BPM"
    },
    "Wind-down": {
      "vibe": "mellow, decompression, mid-tempo 

In [14]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate
import json

llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0.3)

tools = [get_weather, get_temporal_context, get_calendar_load]

# Escape braces in the JSON so LangChain doesn't read them as template variables
TASTE_JSON = json.dumps(TASTE_PROFILE, indent=2).replace("{", "{{").replace("}", "}}")

template = """You are the Mood Cartographer — a contextual music intelligence agent for Siddharth.

Your job: read real-time signals (weather, time, calendar) and predict which mood state Siddharth is most likely in RIGHT NOW, then recommend a playlist concept grounded in his stated taste.

Siddharth's taste profile (use this as ground truth for recommendations):
__TASTE__

Mood states available: Focus, Drive, Workout, Wind-down, Melancholic, Social

Reasoning principles:
- Gather all 3 signals before deciding (weather, temporal, calendar). Don't shortcut.
- Apply context_overrides from the taste profile when relevant.
- Be honest about confidence. If signals conflict, say so.
- The "next_anticipated_state" should be different from current — predict the next likely shift.

You have access to:
{tools}

Use this format:
Question: the input question
Thought: think about which signal to gather first
Action: one of [{tool_names}]
Action Input: input for the tool
Observation: tool result
... (repeat for all 3 signals)
Thought: I have enough signal to synthesize
Final Answer: respond ONLY with a JSON object in this exact schema (no markdown, no code fences, no extra text):
{{
  "current_mood_state": "<one of: Focus|Drive|Workout|Wind-down|Melancholic|Social>",
  "confidence": <float 0.0-1.0>,
  "signals": {{
    "temporal": "<what the time/day tells us>",
    "weather": "<what the weather tells us>",
    "calendar": "<what the calendar tells us>"
  }},
  "reasoning": "<2-3 sentences explaining the synthesis, including any conflicts>",
  "playlist_concept": {{
    "name": "<short evocative name>",
    "vibe": "<one sentence>",
    "tempo_range": "<BPM range>",
    "anchor_artists": ["<2-3 from his taste profile>"]
  }},
  "next_anticipated_state": "<the likely next mood shift and when>"
}}

Question: {input}
Thought:{agent_scratchpad}"""

template = template.replace("__TASTE__", TASTE_JSON)
mood_prompt = PromptTemplate.from_template(template)

mood_agent = create_react_agent(llm, tools, mood_prompt)
mood_executor = AgentExecutor(
    agent=mood_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8,
)

print("✅ Mood Cartographer agent ready")

✅ Mood Cartographer agent ready


In [15]:
result = mood_executor.invoke({
    "input": "What should I listen to right now? Read all 3 signals — weather, time, calendar — then return the JSON."
})

print("\n=== AGENT OUTPUT ===\n")
print(result["output"])



> Entering new AgentExecutor chain...
Thought: I need to gather all 3 signals before making any recommendation. Let me start with the temporal context to understand what time/day it is for Siddharth.

Action: get_temporal_context
Action Input: America/Los_AngelesSunday 10:39 PM (late evening). Weekend: True. Life rhythm: Sunday work-prep.Thought: It's Sunday late evening — the "Sunday_evening" context override says lean toward Focus or Wind-down, and "Sunday work-prep" rhythm reinforces that. Now let me check the weather.

Action: get_weather
Action Input: San FranciscoSan Francisco: 53.2°F, overcast, cloud cover 86%, wind 10.0 mph. Mood descriptor: muted.Thought: Overcast and muted weather — not quite rainy/foggy enough to fully trigger the Melancholic override, but it's close. The "muted" descriptor leans atmospheric. Now let me check the calendar to see what tomorrow looks like.

Action: get_calendar_load
Action Input: 24Error: unsupported type for timedelta hours component: strTh

In [16]:
@tool
def get_calendar_load(hours_ahead: int = 24) -> str:
    """Returns calendar density for the next N hours. Categorizes as 'light', 'moderate', 'heavy'. Lists event titles."""
    try:
        hours_ahead = int(hours_ahead)  # coerce string input from agent
        now = datetime.now(timezone.utc)
        end = now + timedelta(hours=hours_ahead)
        events = calendar.events().list(
            calendarId="primary",
            timeMin=now.isoformat(),
            timeMax=end.isoformat(),
            singleEvents=True,
            orderBy="startTime"
        ).execute().get("items", [])

        count = len(events)
        if count == 0:
            load = "light (no events)"
        elif count <= 2:
            load = "light"
        elif count <= 5:
            load = "moderate"
        else:
            load = "heavy"

        titles = [e.get("summary", "(untitled)") for e in events[:5]]
        return f"Next {hours_ahead}h: {count} events ({load}). Titles: {titles}"
    except Exception as e:
        return f"Error: {e}"

# Rebuild agent with fixed tool
tools = [get_weather, get_temporal_context, get_calendar_load]
mood_agent = create_react_agent(llm, tools, mood_prompt)
mood_executor = AgentExecutor(
    agent=mood_agent, tools=tools, verbose=True,
    handle_parsing_errors=True, max_iterations=8,
)
print("✅ Fixed + rebuilt")

✅ Fixed + rebuilt


In [17]:
result = mood_executor.invoke({
    "input": "What should I listen to right now? Read all 3 signals — weather, time, calendar — then return the JSON."
})

print("\n=== AGENT OUTPUT ===\n")
print(result["output"])



> Entering new AgentExecutor chain...
Thought: I need to gather all 3 signals before making any recommendation. Let me start with the temporal context to understand what time and day it is for Siddharth.

Action: get_temporal_context
Action Input: America/Los_AngelesSunday 10:52 PM (late evening). Weekend: True. Life rhythm: Sunday work-prep.Thought: It's Sunday late evening — the "Sunday_evening" context override says lean toward Focus or Wind-down, and the "Sunday work-prep" life rhythm reinforces that. Now let me check the weather.

Action: get_weather
Action Input: San FranciscoSan Francisco: 52.8°F, overcast, cloud cover 94%, wind 10.0 mph. Mood descriptor: muted.Thought: Overcast and muted weather — while not explicitly rainy/foggy, the heavy cloud cover and muted descriptor nudges toward Melancholic territory. But it's not a hard override since it's not rain/fog specifically. Now let me check the calendar.

Action: get_calendar_load
Action Input: 24Next 24h: 1 events (light). 